# H8: are DKA and the text branch wrong on the same images?
Before building any fusion (late logit fusion, distillation H6, cross-attention H7), check whether two models make **different** mistakes. If they mostly fail on the same test images, no way of combining them can help much.

For every pair of models, on the same training subset (seed) and the same test images, this notebook reports:
- **acc A, acc B** and **oracle**: accuracy if you could always pick whichever of the two is right. This is the ceiling for any combination.
- **headroom** = oracle - best single model. Small headroom (a few points) means fusion cannot gain much.
- **error overlap** (Jaccard of the two error sets) and **P(B wrong | A wrong)**.
- **error correlation**: Cohen's kappa on correct/incorrect. High kappa means they fail together.
- **late fusion**: average of the two models' probabilities, and of their log-probabilities (the blend that worked in the PARA study), with fixed equal weights (no tuning on the test set).
- **McNemar** (exact, discordant pairs pooled over seeds): are A and B really different?

Inputs (produced by the v2 notebooks; nothing is retrained here):
- `mm_dka/dka_preds_v2.csv` (DKA variants)
- `mm_pipeline_busi/preds_v2.csv`, `mm_pipeline_covid_full/preds_v2.csv` (frozen pipeline variants)
- `mm_medsiglip/preds.csv` (MedSigLIP controls)

Text runs are filtered to each dataset's current caption file, so old-question runs never mix in.

In [ ]:
# CELL 0 - Load every model's per-image test predictions into one table
import os, json, numpy as np, pandas as pd
from scipy.stats import binomtest

SOURCES = {   # source name -> (file, dataset name used here, or None when the file has a "dataset" column)
    "dka":       ("./mm_dka/dka_preds_v2.csv", None),
    "pipe_busi": ("./mm_pipeline_busi/preds_v2.csv", "BUSI"),
    "pipe_covid":("./mm_pipeline_covid_full/preds_v2.csv", "COVID_full"),
    "medsiglip": ("./mm_medsiglip/preds.csv", None),
}
CAPTIONS = {}   # dataset -> current caption file (text runs from other caption files are dropped)
for ds, pdir in [("BUSI", "./mm_pipeline_busi"), ("COVID_full", "./mm_pipeline_covid_full")]:
    CAPTIONS[ds] = os.path.basename(json.load(open(f"{pdir}/run_info.json"))["caps_file"])

def read_preds(path):
    """BUSI (3 classes) and COVID (4) share one file, so the header lists p0-p2 while COVID rows carry p3 too."""
    head = open(path, encoding="utf8").readline().strip().split(",")
    base = [c for c in head if not (c[:1] == "p" and c[1:].isdigit())]
    return pd.read_csv(path, names=base + [f"p{k}" for k in range(16)], header=None, skiprows=1).dropna(axis=1, how="all")

frames = []
for src, (path, ds) in SOURCES.items():
    if not os.path.exists(path):
        print(f"missing (run its notebook first): {path}"); continue
    p = read_preds(path)
    if ds: p["dataset"] = ds
    p["text_tag"] = p["text_tag"].fillna("")
    p = p[(p["text_tag"] == "") | (p["text_tag"] == p["dataset"].map(CAPTIONS))]
    prefix = {"dka": "", "medsiglip": "msl:"}.get(src, "pipe:")          # e.g. "pipe:full", "msl:imagenet+text", "dka_text"
    p["model"] = prefix + p["variant"]
    frames.append(p)
P = pd.concat(frames, ignore_index=True)
P["frac"] = P["frac"].round(4)
P = P.drop_duplicates(["dataset", "model", "frac", "seed", "image_id"], keep="last")   # an interrupted run may have been redone
print("predictions:", len(P), "rows")
print(P.groupby(["dataset", "frac"])["model"].unique().apply(lambda m: ", ".join(sorted(m))).to_string())


In [ ]:
# CELL 1 - Pairwise complementarity
def kappa(a, b):
    po = (a == b).mean(); pa, pb = a.mean(), b.mean(); pe = pa * pb + (1 - pa) * (1 - pb)
    return (po - pe) / (1 - pe) if pe < 1 else np.nan

def compare(ds, frac, A, B):
    pc = [c for c in P.columns if c.startswith("p") and c[1:].isdigit()]
    a = P[(P["dataset"] == ds) & (P["frac"] == round(frac, 4)) & (P["model"] == A)].set_index(["seed", "image_id"])
    b = P[(P["dataset"] == ds) & (P["frac"] == round(frac, 4)) & (P["model"] == B)].set_index(["seed", "image_id"])
    j = a.join(b, lsuffix="_a", rsuffix="_b", how="inner")
    if j.empty: return None
    rows = []
    for seed, g in j.groupby(level=0):
        y = g["y_true_a"].values; ca = g["y_pred_a"].values == y; cb = g["y_pred_b"].values == y
        pa = g[[c + "_a" for c in pc]].values.astype(float); pb = g[[c + "_b" for c in pc]].values.astype(float)
        ok = ~np.isnan(pa).all(1)
        avg = ((pa + pb) / 2).argmax(1); logavg = (np.log(pa + 1e-8) + np.log(pb + 1e-8)).argmax(1)
        ea, eb = ~ca, ~cb
        rows.append(dict(seed=seed, n=len(y), acc_a=100 * ca.mean(), acc_b=100 * cb.mean(), oracle=100 * (ca | cb).mean(),
                         both_wrong=100 * (ea & eb).mean(),
                         overlap=(ea & eb).sum() / max(1, (ea | eb).sum()),
                         p_b_wrong_given_a_wrong=(ea & eb).sum() / max(1, ea.sum()),
                         kappa=kappa(ca, cb),
                         late_prob=100 * (avg == y)[ok].mean(), late_logprob=100 * (logavg == y)[ok].mean(),
                         n10=int((ca & ~cb).sum()), n01=int((~ca & cb).sum())))
    r = pd.DataFrame(rows)
    n10, n01 = r["n10"].sum(), r["n01"].sum()
    out = r.drop(columns=["seed", "n", "n10", "n01"]).mean().to_dict()
    out.update(seeds=len(r), headroom=out["oracle"] - max(out["acc_a"], out["acc_b"]),
               late_gain=max(out["late_prob"], out["late_logprob"]) - max(out["acc_a"], out["acc_b"]),
               mcnemar_p=binomtest(int(n10), int(n10 + n01), 0.5).pvalue if n10 + n01 else 1.0, n10=int(n10), n01=int(n01))
    return out

# pairs to check: (A, B, why)
PAIRS = [
    ("dka",       "pipe:full",               "DKA vs frozen ImageNet + local + text"),
    ("dka",       "msl:imagenet+text",       "DKA vs frozen ImageNet + text"),
    ("dka",       "msl:imagenet+medsiglip",  "DKA vs frozen ImageNet + MedSigLIP embedding"),
    ("dka",       "msl:lp_medsiglip",        "DKA vs MedSigLIP linear probe"),
    ("dka_mlp",   "dka_text",                "DKA with vs without text"),
    ("pipe:global+local", "pipe:full",       "frozen: text vs no text"),
    ("msl:imagenet+text", "msl:imagenet+medsiglip", "text vs MedSigLIP embedding (H2)"),
    ("msl:mlp_medsiglip", "msl:text_only",   "MedSigLIP image vs text alone"),
]
table = []
for ds in sorted(P["dataset"].unique()):
    for frac in sorted(P[P["dataset"] == ds]["frac"].unique()):
        for A, B, why in PAIRS:
            r = compare(ds, frac, A, B)
            if r: table.append(dict(dataset=ds, frac=f"{frac*100:g}%", A=A, B=B, why=why, **r))
T = pd.DataFrame(table)
show = ["dataset", "frac", "A", "B", "seeds", "acc_a", "acc_b", "oracle", "headroom", "overlap", "kappa",
        "late_prob", "late_logprob", "late_gain", "n10", "n01", "mcnemar_p"]
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)
print(T[show].round({"acc_a": 1, "acc_b": 1, "oracle": 1, "headroom": 1, "overlap": 2, "kappa": 2,
                     "late_prob": 1, "late_logprob": 1, "late_gain": 1, "mcnemar_p": 4}).to_string(index=False))
os.makedirs("./mm_analysis", exist_ok=True); T.to_csv("./mm_analysis/h8_complementarity.csv", index=False)


In [ ]:
# CELL 2 - Verdict per pair
# headroom < 3 points and late fusion gains < 1 point -> the two models fail on mostly the same images:
#   no fusion / distillation (H6, H7) is likely to stack them.
# headroom >= 5 points and kappa < 0.4 -> they make different mistakes: fusion is worth trying,
#   starting with the free log-prob late fusion before building anything.
for _, r in T.iterrows():
    if r["headroom"] < 3 and r["late_gain"] < 1: verdict = "redundant: same mistakes, skip fusion"
    elif r["headroom"] >= 5 and r["kappa"] < 0.4: verdict = "complementary: fusion worth trying"
    else: verdict = "in between: try free late fusion first"
    print(f"{r['dataset']:<10} {r['frac']:>6}  {r['A']:<18} vs {r['B']:<24} headroom {r['headroom']:4.1f}  "
          f"late fusion {r['late_gain']:+4.1f}  kappa {r['kappa']:.2f}  ->  {verdict}")
